In [1]:
import pandas as pd
import polars as pl
import ScraperFC as sfc

from fantasy_football.constants import PROJECT_ROOT
from fantasy_football.storage.database import get_connection    
from fantasy_football.storage.tables import MINUTES_PREDICTION

In [2]:
tm = sfc.Transfermarkt()
connection = get_connection()

In [3]:
seasons = ["16/17", "17/18", "18/19", "19/20", "20/21", "21/22", "22/23", "23/24", "24/25", "25/26", "26/27"]

In [4]:
def get_player_links(seasons: list[str]) -> list[str]:
    all_player_links = []
    for season in seasons:
        print(f"Getting player links for {season}")
        all_player_links.extend(tm.get_player_links(year=season, league="England Premier League"))
    
    return all_player_links

In [5]:
player_links = get_player_links(seasons)

Getting player links for 16/17
Running


16/17 England Premier League player links: 100%|██████████| 20/20 [00:34<00:00,  1.71s/it]


Getting player links for 17/18


17/18 England Premier League player links: 100%|██████████| 20/20 [00:35<00:00,  1.79s/it]


Getting player links for 18/19


...

24/25 England Premier League player links: 100%|██████████| 20/20 [01:03<00:00,  3.16s/it]


Getting player links for 25/26


25/26 England Premier League player links: 100%|██████████| 20/20 [00:47<00:00,  2.37s/it]


Getting player links for 26/27


26/27 England Premier League player links: 100%|██████████| 20/20 [00:24<00:00,  1.20s/it]


In [12]:
player_links

['https://www.transfermarkt.us/jonathan-calleri/marktwertverlauf/spieler/284727',
 'https://www.transfermarkt.us/ademola-lookman/profil/spieler/406040',
 'https://www.transfermarkt.us/david-nugent/profil/spieler/33963',
 'https://www.transfermarkt.us/luke-amos/profil/spieler/258912',
 'https://www.transfermarkt.us/matt-phillips/profil/spieler/77274',
 'https://www.transfermarkt.us/christian-fuchs/marktwertverlauf/spieler/6636',
 'https://www.transfermarkt.us/fabio-borini/marktwertverlauf/spieler/96754',
 'https://www.transfermarkt.us/will-buckley/profil/spieler/77329',
 'https://www.transfermarkt.us/lukasz-fabianski/marktwertverlauf/spieler/29692',
 'https://www.transfermarkt.us/daniel-amartey/marktwertverlauf/spieler/214056',
 'https://www.transfermarkt.us/wilfried-zaha/profil/spieler/145988',
 'https://www.transfermarkt.us/joel-robles/profil/spieler/101118',
 'https://www.transfermarkt.us/james-wilson/marktwertverlauf/spieler/214104',
 'https://www.transfermarkt.us/dele-alli/profil/s

In [13]:
player_1 = tm.scrape_player(player_links[0])
player_1

,Name,ID,Value,Value last updated,DOB,Age,Height (m),Nationality,Citizenship,Position,Other positions,Team,Last club,Since,Joined,Contract expiration,Market value history,Transfer history
0,Jonathan Calleri,284727,€1.50m,"May 26, 2026","Sep 23, 1993",32,1.8,Argentina,[],Centre-Forward,None,São Paulo,None,None,"Jul 25, 2022","Dec 31, 2028",None,"Empty DataFrame Columns: [Season, Date, Left, ..."


In [8]:
nick_pope_link = next(link for link in player_links if "nick-pope" in link)
nick_pope_link

'https://www.transfermarkt.us/nick-pope/profil/spieler/192080'

In [9]:
nick_pope = tm.scrape_player(nick_pope_link)
nick_pope


,Name,ID,Value,Value last updated,DOB,Age,Height (m),Nationality,Citizenship,Position,Other positions,Team,Last club,Since,Joined,Contract expiration,Market value history,Transfer history
0,Nick Pope,192080,€5.00m,"Jun 3, 2026","Apr 19, 1992",34,1.98,England,[England],Goalkeeper,None,Newcastle,None,None,"Jul 1, 2022","Jun 30, 2027",None,"Empty DataFrame Columns: [Season, Date, Left, ..."


In [10]:
minutes_errors_sql = (PROJECT_ROOT / "queries" / "minutes_errors.sql").read_text()
minutes_errors = connection.sql(minutes_errors_sql).pl()

In [11]:
minutes_errors.filter(
    (pl.col("season") == "2026-27") &
    (pl.col("player_team") == "Newcastle") & 
    (pl.col("position") == "GK")
)

run_id,season,gw,kickoff_time,element,player,player_team,opposition,actual_bucket,actual_minutes,expected_minutes,error,absolute_error,p_zero,p_partial,p_sixty_plus,p_appear,appeared,played_sixty,brier_appear_contribution,brier_sixty_contribution,position,value,value_share_of_team,pos_value_rank,players_same_pos,chance_of_playing_this_round,fit_rivals_same_pos,fit_rivals_ahead,avg_minutes_rolling_5,games_played_this_season,prev_season_minutes,prev_season_start_rate,prev_season_points_per_start,pl_seasons_played,seasons_since_last_pl,age_years,is_pl_newcomer,is_promoted_club
str,str,i64,datetime[μs],i64,str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""da88ecc33f2141ceb9ec4088bb2aee…","""2026-27""",1,2026-08-23 15:30:00,567,"""Lukás Hornícek""","""Newcastle""","""Liverpool""","""60_minutes_plus""",90,8.403811,81.596189,81.596189,0.88473,0.005366,0.109905,0.11527,1,1,0.782747,0.79227,"""GK""",50.0,0.035971,1.0,4.0,100.0,3.0,0.0,null,0.0,null,null,null,0.0,null,24.052019,1.0,0.0
"""da88ecc33f2141ceb9ec4088bb2aee…","""2026-27""",1,2026-08-23 15:30:00,442,"""Nick Pope""","""Newcastle""","""Liverpool""","""0_minutes""",0,71.658537,-71.658537,71.658537,0.041533,0.005034,0.953434,0.958467,0,0,0.91866,0.909036,"""GK""",50.0,0.035971,1.0,4.0,100.0,3.0,0.0,90.0,0.0,2416.0,0.710526,3.555556,8.0,0.0,34.283368,0.0,0.0
"""da88ecc33f2141ceb9ec4088bb2aee…","""2026-27""",1,2026-08-23 15:30:00,444,"""Ewen Jaouen""","""Newcastle""","""Liverpool""","""0_minutes""",0,5.919898,-5.919898,5.919898,0.918685,0.003971,0.077343,0.081315,0,0,0.006612,0.005982,"""GK""",45.0,0.032374,3.0,4.0,100.0,3.0,2.0,null,0.0,null,null,null,0.0,null,20.588638,1.0,0.0
"""da88ecc33f2141ceb9ec4088bb2aee…","""2026-27""",1,2026-08-23 15:30:00,443,"""Mark Gillespie""","""Newcastle""","""Liverpool""","""0_minutes""",0,4.803759,-4.803759,4.803759,0.931488,0.007436,0.061076,0.068512,0,0,0.004694,0.00373,"""GK""",45.0,0.032374,3.0,4.0,100.0,3.0,2.0,0.0,0.0,0.0,0.0,null,5.0,0.0,34.346338,0.0,0.0


In [14]:
connection.close()

In [17]:
output_filepath = PROJECT_ROOT / "data" / "player_links.txt"

with open(output_filepath, "w", encoding="utf-8") as f:
    f.write("\n".join(player_links) + "\n")